# Woche 01: Globales Alignment (Needleman–Wunsch) — Übung

## Lernziele
- Globales Pairwise-Alignment als **dynamische Programmierung** umsetzen (Needleman–Wunsch).
- Verstehen, wie **Scoring** (Match/Mismatch/Gap) biologische Annahmen ausdrückt.
- Laufzeit-/Speichergrenzen (≈ \(O(nm)\)) praktisch erleben.

## Aufgaben
1. Implementiere `compute_dp(...)`: fülle die DP-Matrix für das globale Alignment.
2. Implementiere `traceback(...)`: rekonstruiere ein optimales Alignment.
   - **Tie-Break-Regel** (für deterministische Tests): bevorzuge *Diagonal* > *Up* > *Left*.
3. Implementiere `needleman_wunsch(...)`: nutze DP + Traceback und gib ein `AlignmentResult` zurück.
4. Implementiere `alignment_score(...)`: berechne den Score eines Alignments (zur Selbstkontrolle).

**Erwartung:** Die Implementierung soll klar, pythonic und gut typisiert sein.

---

**Hinweis:** Das Notebook enthält Zellen, die die Übungsdateien (`bioinf_week01.py`, `tests/test_week01.py`) per `%%writefile` in den aktuellen Ordner schreiben. Wenn du die Dateien schon aus dem Repo/ZIP nutzt, kannst du diese Zellen überspringen.

## Setup

In einem Terminal im gleichen Ordner ausführen:

```bash
pip install -r requirements.txt
```

Oder (wenn pytest schon da ist):

```bash
pip install -U pytest
```


In [ ]:
%%writefile bioinf_week01.py
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Tuple


@dataclass(frozen=True)
class ScoreScheme:
    """Einfaches Scoring-Schema für DNA/RNA-Alignments."""
    match: int = 1
    mismatch: int = -1
    gap: int = -1


@dataclass(frozen=True)
class AlignmentResult:
    score: int
    aligned_a: str
    aligned_b: str


def compute_dp(seq_a: str, seq_b: str, scheme: ScoreScheme) -> List[List[int]]:
    """
    Baue die DP-Matrix für Needleman–Wunsch (globales Alignment).

    dp[i][j] = bester Score für seq_a[:i] gegen seq_b[:j]
    """
    raise NotImplementedError


def traceback(seq_a: str, seq_b: str, dp: List[List[int]], scheme: ScoreScheme) -> Tuple[str, str]:
    """
    Rekonstruiere EIN optimales Alignment aus dp.

    Tie-Break (wenn mehrere Moves gleich gut sind):
    Diagonal > Up > Left
    """
    raise NotImplementedError


def alignment_score(aligned_a: str, aligned_b: str, scheme: ScoreScheme) -> int:
    """Berechne den Alignment-Score eines bereits ausgerichteten Paars."""
    raise NotImplementedError


def needleman_wunsch(seq_a: str, seq_b: str, scheme: ScoreScheme = ScoreScheme()) -> AlignmentResult:
    """High-level Wrapper: DP bauen, traceback, Score zurückgeben."""
    dp = compute_dp(seq_a, seq_b, scheme)
    aligned_a, aligned_b = traceback(seq_a, seq_b, dp, scheme)
    score = dp[len(seq_a)][len(seq_b)]
    return AlignmentResult(score=score, aligned_a=aligned_a, aligned_b=aligned_b)


In [ ]:
%%writefile tests/test_week01.py
from __future__ import annotations

from pathlib import Path
import sys

# Allow `pytest` from any working directory by adding the week folder to sys.path
sys.path.append(str(Path(__file__).resolve().parents[1]))

from bioinf_week01 import ScoreScheme, needleman_wunsch, alignment_score, compute_dp


def _is_valid_alignment(original: str, aligned: str) -> bool:
    return original == aligned.replace("-", "")


def test_global_alignment_score_matches_dp():
    scheme = ScoreScheme(match=1, mismatch=-1, gap=-1)
    a = "GATTACA"
    b = "GCATGCU"
    dp = compute_dp(a, b, scheme)
    res = needleman_wunsch(a, b, scheme)
    assert res.score == dp[len(a)][len(b)]
    assert _is_valid_alignment(a, res.aligned_a)
    assert _is_valid_alignment(b, res.aligned_b)
    assert alignment_score(res.aligned_a, res.aligned_b, scheme) == res.score


def test_empty_sequences():
    scheme = ScoreScheme(match=2, mismatch=-1, gap=-2)
    res = needleman_wunsch("", "", scheme)
    assert res.score == 0
    assert res.aligned_a == ""
    assert res.aligned_b == ""


def test_one_empty_sequence():
    scheme = ScoreScheme(match=2, mismatch=-1, gap=-2)
    res = needleman_wunsch("ACG", "", scheme)
    assert res.score == 3 * scheme.gap
    assert res.aligned_a.replace("-", "") == "ACG"
    assert res.aligned_b == "---"


## Tests ausführen

```bash
pytest -q
```

Wenn alle Tests grün sind, bist du fertig.